# Step 3: Comprehensive Exploratory Data Analysis (EDA)
In this notebook, we perform a deep dive into our Target Variable using state-of-the-art profiling libraries: `ydata-profiling`, `Sweetviz`, and `D-Tale`.

**Important Memory Strategy:** 
Since the raw ER datasets contain over a hundred million records (which would crash these libraries in RAM), we first use DuckDB to aggregate the data at the daily level per hospital. This brings the dataset to a highly informative and memory-safe size for profiling.

In [1]:
import duckdb
import pandas as pd
import sweetviz as sv
from ydata_profiling import ProfileReport
import dtale
import warnings
warnings.filterwarnings('ignore')

print("=== EXTRACTING AGGREGATED TARGET VARIABLE VIA DUCKDB ===")

# Aggregate to Daily Respiratory Urgencies per Hospital
query_eda = """
SELECT 
    fecha,
    IdEstablecimiento,
    NEstablecimiento,
    SUM(Total) as Total_Respiratorias,
    SUM(Menores_1) as Menores_1,
    SUM(De_1_a_4) as De_1_a_4,
    SUM(De_5_a_14) as De_5_a_14,
    SUM(De_15_a_64) as De_15_a_64,
    SUM(De_65_y_mas) as Mayores_65
FROM read_parquet('../data/processed/urgencias_parquet/*.parquet')
WHERE GlosaCausa LIKE '%RESPIRATORIO%'
GROUP BY fecha, IdEstablecimiento, NEstablecimiento
"""

df_eda = duckdb.query(query_eda).df()

# Format Date
df_eda['fecha'] = pd.to_datetime(df_eda['fecha'], format='%d/%m/%Y', errors='coerce')
df_eda = df_eda.dropna(subset=['fecha']).sort_values('fecha')

print(f"Extracted {len(df_eda)} daily aggregated records for profiling.")
display(df_eda.head())

C:\Users\Usuario\AppData\Local\Temp\ipykernel_21336\531674180.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


=== EXTRACTING AGGREGATED TARGET VARIABLE VIA DUCKDB ===


Extracted 1485513 daily aggregated records for profiling.


,fecha,IdEstablecimiento,NEstablecimiento,Total_Respiratorias,Menores_1,De_1_a_4,De_5_a_14,De_15_a_64,Mayores_65
967098,2017-01-01,18-106,Hospital de Lota,16.0,4.0,1.0,0.0,8.0,3.0
949649,2017-01-01,15-800,SAPU-Enrique Dintrans,20.0,0.0,4.0,6.0,10.0,0.0
105010,2017-01-01,23-800,SAPU-Dr. Pedro Jáuregui,27.0,0.0,7.0,3.0,17.0,0.0
1317307,2017-01-01,24-380,Centro de Salud Familiar Río Negro Hornopirén,4.0,0.0,1.0,1.0,2.0,0.0
242345,2017-01-01,05-308,Consultorio Punitaqui,0.0,0.0,0.0,0.0,0.0,0.0


### 1. Sweetviz Profiling
Sweetviz creates highly visual and comparative HTML reports.

In [2]:
print("\n=== GENERATING SWEETVIZ REPORT ===")
sv_report = sv.analyze(df_eda)
sv_report.show_html('Sweetviz_EDA_Report.html', open_browser=False)
print("Sweetviz report successfully saved to 'Sweetviz_EDA_Report.html'")


=== GENERATING SWEETVIZ REPORT ===


                                             |          | [  0%]   00:00 -> (? left)

Report Sweetviz_EDA_Report.html was generated.
Sweetviz report successfully saved to 'Sweetviz_EDA_Report.html'


### 2. YData Profiling
Generates a comprehensive standardized statistical report.

In [3]:
print("\n=== GENERATING YDATA-PROFILING REPORT ===")
profile = ProfileReport(df_eda, title="Respiratory Urgencies Profiling Report", explorative=True, minimal=True)
profile.to_file("YData_EDA_Report.html")
print("YData Profiling report successfully saved to 'YData_EDA_Report.html'")


=== GENERATING YDATA-PROFILING REPORT ===


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

 11%|█         | 1/9 [00:01<00:15,  1.89s/it]

100%|██████████| 9/9 [00:01<00:00,  4.72it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

YData Profiling report successfully saved to 'YData_EDA_Report.html'


### 3. D-Tale Interactive Exploration
Uncomment and run the cell below in your local Jupyter environment to launch the D-Tale GUI.

In [4]:
# d = dtale.show(df_eda)
# d.open_browser()